## Unzip folder with the data

This folder contains the following

*   Training data of 10k sampled from 1 million patients dataset provided by authors
*   Generated patient sequences
*   OMOP format synthetic dataset
*   Trained model tensors
*   Concept, relationship and acestor tables for mapping



In [ ]:
import zipfile
from pathlib import Path

zip_path = Path("/content/drive/MyDrive/synthetic_data_generated_10k/omop_synthea_sample.zip")
out_dir  = zip_path.parent  # same directory

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(out_dir)

print("Extracted to:", out_dir)
print("Top-level contents:")
for p in sorted(out_dir.iterdir()):
    print(" -", p.name)


In [ ]:
import os, glob, math, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, Markdown

BASE_DIR = "/content/drive/MyDrive/synthetic_data_generated_10k/content/omop_synthea_sample/"
RESTORED_DIR = os.path.join(BASE_DIR, "cehrgpt/synthetic_data/top_p9500_temp_9000_repetition_penalty_10500/restored_omop")
SEQ_DIR = os.path.join(BASE_DIR, "cehrgpt/synthetic_data/top_p9500_temp_9000_repetition_penalty_10500/generated_sequences")

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

def md(s): display(Markdown(s))
def head(df, n=5, title=None):
    if title: md(f"### {title}")
    display(df.head(n))

def read_parquets_in_dir(folder):
    files = sorted(glob.glob(os.path.join(folder, "*.parquet")))
    if not files:
        return pd.DataFrame()
    return pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)

def safe_to_datetime(s):
    return pd.to_datetime(s, errors="coerce")

md("✅ Chunk 1 loaded.")


✅ Chunk 1 loaded.

In [ ]:
TABLES = ["person", "visit_occurrence", "condition_occurrence", "drug_exposure", "procedure_occurrence", "measurement", "death"]

tables = {}
for t in TABLES:
    p = os.path.join(RESTORED_DIR, t)
    if os.path.isdir(p):
        df = read_parquets_in_dir(p)
        tables[t] = df
        md(f"- **{t}**: {df.shape[0]:,} rows × {df.shape[1]} cols")
    else:
        tables[t] = pd.DataFrame()
        md(f"- **{t}**: (missing folder)")

# quick peek
for t in ["person","visit_occurrence","condition_occurrence","drug_exposure","procedure_occurrence"]:
    if len(tables[t])>0:
        head(tables[t], 3, f"{t} sample")


In [ ]:
seq_files = sorted(glob.glob(os.path.join(SEQ_DIR, "*.parquet")))
md(f"Found **{len(seq_files)}** generated_sequences parquet files.")

MAX_SEQ_FILES = 8         # adjust
MAX_ROWS_PER_FILE =10000   # adjust

picked = seq_files[:MAX_SEQ_FILES]
seq_parts = []
for f in picked:
    df = pd.read_parquet(f)
    if len(df) > MAX_ROWS_PER_FILE:
        df = df.sample(MAX_ROWS_PER_FILE, random_state=RANDOM_SEED)
    df["__source_file"] = os.path.basename(f)
    seq_parts.append(df)

gen_df = pd.concat(seq_parts, ignore_index=True) if seq_parts else pd.DataFrame()
md(f"Loaded sequences: **{gen_df.shape[0]:,}** rows × {gen_df.shape[1]} cols from {len(picked)} files.")
head(gen_df, 3, "generated_sequences sample")


In [ ]:
# ✅ Fixed concept-table loader for your setup:
# Concept tables live under: /content/drive/MyDrive/omop_synthea_sample/
# and each is a *directory* containing parquet part files (not a single .parquet file).

VOCAB_ROOT = "/content/drive/MyDrive/synthetic_data_generated_10k/content/omop_synthea_sample"  # <-- your vocab root

def find_table_path(name: str):
    """
    Return either:
      - a directory path containing parquet files (preferred), OR
      - a single .parquet / .csv file path
    Searches VOCAB_ROOT first, then BASE_DIR/RESTORED_DIR.
    """
    roots = [VOCAB_ROOT, BASE_DIR, RESTORED_DIR]

    # 1) directory named exactly like the table (e.g., .../concept/)
    for root in roots:
        cand = os.path.join(root, name)
        if os.path.isdir(cand):
            pq = glob.glob(os.path.join(cand, "*.parquet"))
            if pq:
                return cand  # directory with parquet parts

    # 2) sometimes nested directories
    for root in roots:
        hits = glob.glob(os.path.join(root, f"**/{name}"), recursive=True)
        for h in sorted(hits):
            if os.path.isdir(h):
                pq = glob.glob(os.path.join(h, "*.parquet"))
                if pq:
                    return h

    # 3) single parquet/csv files (fallback)
    patterns = []
    for root in roots:
        patterns.extend([
            os.path.join(root, f"**/{name}.parquet"),
            os.path.join(root, f"**/{name}.csv"),
        ])

    for pat in patterns:
        hits = glob.glob(pat, recursive=True)
        if hits:
            return sorted(hits)[0]

    return None


def load_table_auto(name: str):
    path = find_table_path(name)
    if not path:
        md(f"- **{name}**: not found (searched {VOCAB_ROOT}, BASE_DIR, RESTORED_DIR)")
        return pd.DataFrame()

    # directory of parquet parts
    if os.path.isdir(path):
        files = sorted(glob.glob(os.path.join(path, "*.parquet")))
        df = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True) if files else pd.DataFrame()
        md(f"- **{name}**: {df.shape[0]:,} rows × {df.shape[1]} cols (parquet dir: `{path}`; files={len(files)})")
        return df

    # single parquet/csv file
    if path.endswith(".parquet"):
        df = pd.read_parquet(path)
    else:
        df = pd.read_csv(path)

    md(f"- **{name}**: {df.shape[0]:,} rows × {df.shape[1]} cols (file: `{path}`)")
    return df


# ---- Load vocab tables ----
concept = load_table_auto("concept")
concept_ancestor = load_table_auto("concept_ancestor")
concept_relationship = load_table_auto("concept_relationship")

# ---- Build concept lookup ----
concept_lu = {}
if len(concept) > 0 and "concept_id" in concept.columns:
    name_col = "concept_name" if "concept_name" in concept.columns else None
    dom_col  = "domain_id" if "domain_id" in concept.columns else None
    cls_col  = "concept_class_id" if "concept_class_id" in concept.columns else None

    cols = ["concept_id"] + [c for c in [name_col, dom_col, cls_col] if c]
    cdf = concept[cols].drop_duplicates("concept_id")

    for _, r in cdf.iterrows():
        try:
            cid = int(r["concept_id"])
        except Exception:
            continue
        concept_lu[cid] = {
            "name": r.get(name_col, None) if name_col else None,
            "domain": r.get(dom_col, None) if dom_col else None,
            "class": r.get(cls_col, None) if cls_col else None,
        }

md(f"✅ Concept lookup built for **{len(concept_lu):,}** concept_ids.")


In [ ]:
concept_relationship.sample(10)

In [ ]:
person = tables["person"]
visit  = tables["visit_occurrence"]
cond   = tables["condition_occurrence"]
drug   = tables["drug_exposure"]
proc   = tables["procedure_occurrence"]

md("## Dashboard counts")

if len(person)>0:
    md(f"- Patients: **{person['person_id'].nunique():,}**")
if len(visit)>0:
    md(f"- Visits: **{len(visit):,}** (unique patients: {visit['person_id'].nunique():,})")
if len(cond)>0:
    md(f"- Conditions: **{len(cond):,}** (unique patients: {cond['person_id'].nunique():,})")
if len(drug)>0:
    md(f"- Drugs: **{len(drug):,}** (unique patients: {drug['person_id'].nunique():,})")
if len(proc)>0:
    md(f"- Procedures: **{len(proc):,}** (unique patients: {proc['person_id'].nunique():,})")

# counts per person
def plot_counts_per_person(df, person_col="person_id", title="Counts per person"):
    if len(df)==0:
        md(f"*(skip: no data for {title})*");
        return
    c = df.groupby(person_col).size()
    plt.figure()
    plt.hist(c, bins=30)
    plt.title(title)
    plt.xlabel("Events per person")
    plt.ylabel("Number of persons")
    plt.show()
    display(c.describe(percentiles=[.5,.75,.9,.95,.99]).to_frame("value"))

plot_counts_per_person(visit, title="Visits per person")
plot_counts_per_person(cond, title="Conditions per person")
plot_counts_per_person(drug, title="Drug exposures per person")
plot_counts_per_person(proc, title="Procedures per person")

# visit spans
if len(visit)>0 and "visit_start_date" in visit.columns:
    v = visit.copy()
    v["visit_start_date"] = safe_to_datetime(v["visit_start_date"])
    v["visit_end_date"]   = safe_to_datetime(v.get("visit_end_date", pd.NaT))
    spans = v.groupby("person_id").agg(
        first_visit=("visit_start_date","min"),
        last_visit=("visit_start_date","max"),
        n_visits=("visit_occurrence_id","count")
    )
    spans["span_days"] = (spans["last_visit"] - spans["first_visit"]).dt.days
    md("### Care span (days between first & last visit)")
    display(spans["span_days"].describe(percentiles=[.5,.75,.9,.95,.99]).to_frame("span_days"))


In [ ]:
def attach_concept_name(series_of_ids):
    # returns a series of names if we have vocab
    if not concept_lu:
        return series_of_ids.astype(str)
    def _name(x):
        try:
            x = int(x)
        except:
            return str(x)
        info = concept_lu.get(x, {})
        nm = info.get("name")
        dom = info.get("domain")
        return f"{x} | {nm}" if nm else f"{x}"
    return series_of_ids.map(_name)

def top_k(df, id_col, k=20, title=None):
    if len(df)==0 or id_col not in df.columns:
        md(f"*(skip: {title or id_col} missing)*")
        return
    counts = df[id_col].value_counts().head(k).to_frame("count")
    counts.insert(0, "concept", attach_concept_name(counts.index.to_series()).values)
    if title: md(f"### {title}")
    display(counts)

top_k(tables["condition_occurrence"], "condition_concept_id", 25, "Top condition concepts")
top_k(tables["drug_exposure"], "drug_concept_id", 25, "Top drug concepts")
top_k(tables["procedure_occurrence"], "procedure_concept_id", 25, "Top procedure concepts")
top_k(tables["visit_occurrence"], "visit_concept_id", 25, "Top visit concepts")


In [ ]:
def concept_label(cid):
    if pd.isna(cid): return "NA"
    try: cid = int(cid)
    except: return str(cid)
    if cid in concept_lu and concept_lu[cid].get("name"):
        return f"{concept_lu[cid]['name']} ({cid})"
    return str(cid)

def patient_narrative(person_id):
    lines = []
    # demographics
    if len(person)>0:
        p = person[person["person_id"]==person_id]
        if len(p)>0:
            r = p.iloc[0]
            yob = r.get("year_of_birth", None)
            gender = concept_label(r.get("gender_concept_id", None))
            race = concept_label(r.get("race_concept_id", None))
            eth = concept_label(r.get("ethnicity_concept_id", None))
            lines.append(f"**Patient {person_id}** | YOB={yob} | gender={gender} | race={race} | ethnicity={eth}")
        else:
            lines.append(f"**Patient {person_id}**")
    else:
        lines.append(f"**Patient {person_id}**")

    # visits
    if len(visit)>0:
        v = visit[visit["person_id"]==person_id].copy()
        if len(v)>0:
            v["visit_start_date"] = safe_to_datetime(v["visit_start_date"])
            v = v.sort_values("visit_start_date")
            lines.append(f"- Visits: {len(v)} spanning {v['visit_start_date'].min().date()} → {v['visit_start_date'].max().date()}")
            # top visit types
            vc = v["visit_concept_id"].value_counts().head(5)
            lines.append("- Common visit types: " + "; ".join([concept_label(i) + f"×{c}" for i,c in vc.items()]))

    # conditions
    if len(cond)>0:
        c = cond[cond["person_id"]==person_id].copy()
        if len(c)>0:
            c["condition_start_date"] = safe_to_datetime(c["condition_start_date"])
            c = c.sort_values("condition_start_date")
            topc = c["condition_concept_id"].value_counts().head(8)
            lines.append("- Conditions (top): " + "; ".join([concept_label(i) + f"×{cc}" for i,cc in topc.items()]))

    # drugs
    if len(drug)>0:
        d = drug[drug["person_id"]==person_id].copy()
        if len(d)>0:
            d["drug_exposure_start_date"] = safe_to_datetime(d["drug_exposure_start_date"])
            d = d.sort_values("drug_exposure_start_date")
            topd = d["drug_concept_id"].value_counts().head(8)
            lines.append("- Drugs (top): " + "; ".join([concept_label(i) + f"×{cc}" for i,cc in topd.items()]))

    # procedures
    if len(proc)>0:
        pr = proc[proc["person_id"]==person_id].copy()
        if len(pr)>0:
            pr["procedure_date"] = safe_to_datetime(pr["procedure_date"])
            pr = pr.sort_values("procedure_date")
            topp = pr["procedure_concept_id"].value_counts().head(8)
            lines.append("- Procedures (top): " + "; ".join([concept_label(i) + f"×{cc}" for i,cc in topp.items()]))

    return "\n".join(lines)

# pick patients
patient_ids = sorted(set(person["person_id"].unique())) if len(person)>0 else sorted(set(
    pd.concat([visit.get("person_id",pd.Series(dtype=int)),
               cond.get("person_id",pd.Series(dtype=int)),
               drug.get("person_id",pd.Series(dtype=int)),
               proc.get("person_id",pd.Series(dtype=int))]).dropna().unique()
))

N_PATIENTS = 8
sample_ids = random.sample(list(patient_ids), min(N_PATIENTS, len(patient_ids)))

md("## Patient narratives")
for pid in sample_ids:
    md(patient_narrative(pid))
    md("---")


In [ ]:
def monthly_counts(df, person_id, date_col):
    if len(df)==0 or date_col not in df.columns:
        return pd.Series(dtype=int)
    x = df[df["person_id"]==person_id].copy()
    if len(x)==0: return pd.Series(dtype=int)
    x[date_col] = safe_to_datetime(x[date_col])
    x = x.dropna(subset=[date_col])
    if len(x)==0: return pd.Series(dtype=int)
    return x.set_index(date_col).resample("M").size()

md("## Timeline summaries (monthly event counts)")

for pid in sample_ids[:5]:
    v = monthly_counts(visit, pid, "visit_start_date")
    c = monthly_counts(cond,  pid, "condition_start_date")
    d = monthly_counts(drug,  pid, "drug_exposure_start_date")
    p = monthly_counts(proc,  pid, "procedure_date")

    idx = v.index.union(c.index).union(d.index).union(p.index)
    out = pd.DataFrame({
        "visits": v.reindex(idx, fill_value=0),
        "conditions": c.reindex(idx, fill_value=0),
        "drugs": d.reindex(idx, fill_value=0),
        "procedures": p.reindex(idx, fill_value=0),
    }).sort_index()

    md(f"### Patient {pid}")
    display(out.tail(24))

    plt.figure()
    for col in out.columns:
        plt.plot(out.index, out[col].values, label=col)
    plt.title(f"Patient {pid}: monthly events")
    plt.xlabel("Month")
    plt.ylabel("Count")
    plt.legend()
    plt.show()


In [ ]:
from collections import defaultdict

md("## Cohort feature vectors")

TOPK = 50

def top_ids(df, id_col, k):
    if len(df)==0 or id_col not in df.columns:
        return []
    return df[id_col].value_counts().head(k).index.tolist()

top_cond = top_ids(cond, "condition_concept_id", TOPK)
top_drug = top_ids(drug, "drug_concept_id", TOPK)
top_proc = top_ids(proc, "procedure_concept_id", TOPK)

md(f"- Using TopK={TOPK}: conditions={len(top_cond)}, drugs={len(top_drug)}, procedures={len(top_proc)}")

# base patient frame
if len(person)>0:
    feat = person[["person_id","year_of_birth","gender_concept_id"]].copy()
else:
    feat = pd.DataFrame({"person_id": patient_ids})

# yob buckets
if "year_of_birth" in feat.columns:
    yob = feat["year_of_birth"]
    feat["yob_bucket"] = pd.cut(yob, bins=[1900,1940,1950,1960,1970,1980,1990,2000,2010,2030], right=False)
    yob_dum = pd.get_dummies(feat["yob_bucket"], prefix="yob", dummy_na=True)
    feat = pd.concat([feat.drop(columns=["yob_bucket"]), yob_dum], axis=1)

# gender one-hot (concept id)
if "gender_concept_id" in feat.columns:
    g_dum = pd.get_dummies(feat["gender_concept_id"].fillna(-1).astype(int), prefix="gender")
    feat = pd.concat([feat.drop(columns=["gender_concept_id"]), g_dum], axis=1)

feat = feat.set_index("person_id")

# add counts for top concepts
def add_counts(base, df, person_col, concept_col, ids, prefix):
    if len(df)==0 or concept_col not in df.columns:
        return base
    sub = df[df[concept_col].isin(ids)]
    ct = sub.groupby([person_col, concept_col]).size().unstack(fill_value=0)
    ct = ct.reindex(columns=ids, fill_value=0)
    ct.columns = [f"{prefix}_{int(c)}" for c in ct.columns]
    return base.join(ct, how="left").fillna(0)

feat = add_counts(feat, cond, "person_id","condition_concept_id", top_cond, "cond")
feat = add_counts(feat, drug, "person_id","drug_concept_id", top_drug, "drug")
feat = add_counts(feat, proc, "person_id","procedure_concept_id", top_proc, "proc")

md(f"Feature matrix: **{feat.shape[0]:,} patients × {feat.shape[1]:,} features**")
head(feat.reset_index(), 5, "Feature matrix sample")


In [ ]:
md("## Sequence realism checks")

if len(gen_df)==0:
    md("*(skip: gen_df empty)*")
else:
    # normalize concept_ids into python list
    def as_list(x):
        if isinstance(x, (list, tuple, np.ndarray)):
            return list(x)
        # sometimes it's a string repr
        if isinstance(x, str):
            return [t for t in x.strip("[]").replace("'", "").split() if t]
        return [x]

    seq_tokens = gen_df["concept_ids"].apply(as_list)

    # flatten sample
    MAX_SEQ_ROWS = min(1500, len(seq_tokens))
    sample_tokens = []
    for toks in seq_tokens.sample(MAX_SEQ_ROWS, random_state=RANDOM_SEED):
        sample_tokens.extend([str(t) for t in toks])

    tok_ser = pd.Series(sample_tokens)

    # marker frequencies
    for marker in ["[VS]","[VE]"]:
        md(f"- `{marker}` frequency: **{(tok_ser==marker).mean():.3f}** (share of tokens)")

    # year/age token checks
    md(f"- `year:` tokens share: **{tok_ser.str.startswith('year:').mean():.3f}**")
    md(f"- `age:` tokens share: **{tok_ser.str.startswith('age:').mean():.3f}**")

    # numeric concept tokens
    is_numeric = tok_ser.str.fullmatch(r"\d+").fillna(False)
    md(f"- Pure numeric tokens share: **{is_numeric.mean():.3f}**")

    numeric_ids = tok_ser[is_numeric].astype(int)
    if len(numeric_ids) > 0 and len(concept_lu)>0:
        in_vocab = numeric_ids.map(lambda x: x in concept_lu)
        md(f"- Numeric tokens in concept vocab: **{in_vocab.mean():.3f}**")

        unknown = numeric_ids[~in_vocab]
        unk_counts = unknown.value_counts().head(25).to_frame("count")
        unk_counts.insert(0, "concept_id", unk_counts.index.astype(int))
        md("### Top unknown numeric tokens (not in concept table)")
        display(unk_counts.reset_index(drop=True))
    else:
        md("*(No concept vocab loaded OR no numeric tokens found.)*")

    # distribution of sequence lengths
    lengths = seq_tokens.apply(len)
    plt.figure()
    plt.hist(lengths, bins=30)
    plt.title("Sequence length distribution (tokens per row)")
    plt.xlabel("Tokens")
    plt.ylabel("Rows")
    plt.show()
    display(lengths.describe(percentiles=[.5,.75,.9,.95,.99]).to_frame("len_tokens"))
